# Intelligent Personal Finance DSS — Pipeline Huấn luyện & Đánh giá Học máy (Nhân — N1–N4)

Notebook **mỏng (thin wrapper)** gọi trực tiếp package chuẩn `finance_ml` để đảm bảo **không lệch đặc trưng (zero train-serving skew)** và **khớp phiên bản thư viện** với FastAPI backend của Khoa.

In [ ]:
# Cell 1: Mount Google Drive, giải nén mã nguồn finance_ml & khóa phiên bản thư viện
import os, shutil, subprocess, sys, zipfile
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR = Path('/content/drive/MyDrive/PTHTTM_Nhan_ML_Colab')
    WORK_DIR = Path('/content/finance_ml_work')
    if WORK_DIR.exists():
        shutil.rmtree(WORK_DIR)
    WORK_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(DRIVE_DIR / 'finance_ml_colab_package.zip') as zf:
        zf.extractall(WORK_DIR)
    os.chdir(WORK_DIR)
    os.environ['PYTHONPATH'] = f"{WORK_DIR / 'src'}:{WORK_DIR}"
    sys.path.insert(0, str(WORK_DIR / 'src'))
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-ml.txt'])
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '-e', '.'])
else:
    if not Path('pyproject.toml').exists() and Path('../pyproject.toml').exists():
        os.chdir('..')
    DRIVE_DIR = Path('./artifacts/colab_drive_preview')
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)

import numpy as np, pandas as pd, sklearn
gpu_check = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True)
print('GPU Runtime:', gpu_check.stdout.strip() if gpu_check.returncode == 0 else 'CPU Runtime (Lưu ý: các thuật toán scikit-learn thực thi trên CPU)')
print(f'Pinned versions -> Python {sys.version.split()[0]}, scikit-learn {sklearn.__version__}, pandas {pd.__version__}, numpy {np.__version__}')

In [ ]:
# Cell 2: Sinh dữ liệu mô phỏng VND (seed=42) & hiển thị biểu đồ EDA
import subprocess, sys
from IPython.display import Image, Markdown, display

subprocess.check_call([sys.executable, 'scripts/generate_data.py'])
# Nếu bạn đặt thêm daily_household_transactions.csv trên Drive, tự động copy vào data/raw/
kaggle_on_drive = DRIVE_DIR / 'daily_household_transactions.csv'
if kaggle_on_drive.exists():
    Path('data/raw').mkdir(parents=True, exist_ok=True)
    shutil.copy2(kaggle_on_drive, Path('data/raw/daily_household_transactions.csv'))
subprocess.check_call([sys.executable, 'scripts/prepare_public_data.py'])
subprocess.check_call([sys.executable, 'scripts/run_eda.py', '--dataset', 'synthetic'])

for fig_name in ['daily_spending_trend.png', 'category_share.png', 'amount_log_distribution.png']:
    display(Image(filename=f'artifacts/synthetic/figures/{fig_name}'))

In [ ]:
# Cell 3: Huấn luyện mô hình F3 (Baseline, LinearRegression, RandomForestRegressor) & Đánh giá Validation + Rolling CV
import json, subprocess, sys
from IPython.display import Image, display

subprocess.check_call([sys.executable, 'scripts/train.py', '--dataset', 'synthetic', '--version', 'v1.0.0'])
subprocess.check_call([sys.executable, 'scripts/evaluate.py', '--dataset', 'synthetic'])

val_metrics = json.loads(Path('artifacts/synthetic/metrics/validation_metrics.json').read_text(encoding='utf-8'))
print('Selected F3 Model on Validation:', val_metrics['selected_forecast_model'])
display(pd.DataFrame(val_metrics['forecast_validation']).T[['mae_vnd', 'rmse_vnd', 'n_samples', 'improvement_vs_baseline_pct', 'meets_10pct_target']])
display(Image(filename='artifacts/synthetic/figures/forecast_val_comparison.png'))

In [ ]:
# Cell 4: Kết quả F4 (Isolation Forest), Lý do cảnh báo tiếng Việt & Benchmark chèn bất thường mô phỏng
import json
from IPython.display import Image, display

print('F4 Validation Alert Summary:', json.dumps(val_metrics['anomaly_validation'], indent=2, ensure_ascii=False))
inj = json.loads(Path('artifacts/synthetic/metrics/synthetic_anomaly_eval.json').read_text(encoding='utf-8'))
print('Synthetic Injection Benchmark (tách riêng):', json.dumps(inj, indent=2, ensure_ascii=False))
display(Image(filename='artifacts/synthetic/figures/anomaly_score_distribution.png'))

review_df = pd.read_csv('artifacts/synthetic/anomaly_manual_review_validation.csv')
display(review_df.head(10))

In [ ]:
# Cell 5: Xuất Bundle mô hình (.joblib + metadata.json), Báo cáo & CSV rà soát về Google Drive
# Lưu ý: Tập TEST vẫn được GIỮ KHÓA theo quy tắc 2A để bảo toàn tính độc lập.
out_drive = DRIVE_DIR / 'outputs'
out_drive.mkdir(parents=True, exist_ok=True)
shutil.copytree('artifacts/synthetic', out_drive / 'synthetic', dirs_exist_ok=True)
shutil.copy2('docs/eda_report.md', out_drive / 'eda_report.md')
shutil.copy2('docs/model_card.md', out_drive / 'model_card.md')
print('Đã lưu toàn bộ model bundle, biểu đồ và báo cáo sang:', out_drive)

# CHỈ BỎ COMMENT DÒNG DƯỚI ĐÂY ĐÚNG 1 LẦN KHI ĐÃ CHỐT CẤU HÌNH CUỐI CÙNG VỚI KHOA (NGÀY 14):
# subprocess.check_call([sys.executable, 'scripts/evaluate.py', '--dataset', 'synthetic', '--unlock-test', '--reason', 'One-shot final test evaluation'])

In [ ]:
# Cell 6 (Demo): Kiểm tra trực tiếp 1 giao dịch cụ thể (giống luồng UI nhấp vào giao dịch -> FastAPI gọi detect_anomalies)
import json
from datetime import date
from finance_ml.inference.loader import ModelBundleLoader
from finance_ml.inference.predictor import detect_anomalies, predict_next_7d

bundle = ModelBundleLoader(Path('artifacts/models/active.json')).load()
tx_df = pd.read_csv('data/processed/synthetic/transactions.csv')

# Thử thêm 1 giao dịch đột biến 9.5 triệu VND ở danh mục shopping cho user_001 để xem model chấm điểm & giải thích
test_tx = pd.DataFrame([{
    'transaction_id': 'demo_check_001',
    'user_id': 'user_001',
    'timestamp': '2026-09-15T21:30:00+07:00',
    'type': 'expense',
    'amount_vnd': 9500000.0,
    'category': 'shopping',
    'is_internal_transfer': False,
    'note': 'Thử kiểm tra giao dịch bất thường'
}])
combined = pd.concat([tx_df, test_tx], ignore_index=True)

res_f4 = detect_anomalies(combined, user_id='user_001', bundle=bundle, target_transaction_ids=['demo_check_001'])
print('=== KẾT QUẢ KIỂM TRA 1 GIAO DỊCH (F4) ===')
print(json.dumps(res_f4.model_dump(mode='json'), indent=2, ensure_ascii=False))

res_f3 = predict_next_7d(tx_df, user_id='user_001', as_of_date=date(2026, 9, 15), bundle=bundle)
print('\n=== KẾT QUẢ DỰ BÁO 7 NGÀY TỚI (F3) ===')
print(json.dumps(res_f3.model_dump(mode='json'), indent=2, ensure_ascii=False))